# Эмбеддинги отзывов на трансформере: токены, маска, пулинг, кэш

Локальный блокнот к урокам главы 13 «Эмбеддинги: векторы слов и отзывов» (разделы 3–5) и «Внутри трансформера: токены, маска, пулинг» (разделы 1–2). В браузере PyTorch не работает, поэтому здесь — то, что на сайте показано на игрушечных массивах numpy:

1. токенизатор настоящей модели: подслова, `[CLS]`/`[SEP]`, дополнение и маска;
2. векторы токенов и пулинг с маской — и проверка, что вектор отзыва не зависит от соседей по пакету;
3. кодирование всех отзывов пакетами с **кэшем на диске**: второй запуск — секунды вместо минут;
4. классификатор тональности на эмбеддингах против TF-IDF;
5. поиск похожих отзывов по смыслу.

**Модель.** `cointegrated/rubert-tiny2` — маленький русский BERT (около 120 МБ, вектор из 312 чисел). Скачивается с Hugging Face при первом запуске и дальше берётся из локального кэша (`~/.cache/huggingface`).

**Как запустить.**

```sh
uv add transformers torch          # GPU не нужен
# на Linux без видеокарты torch меньше в CPU-сборке:
# uv pip install torch --index-url https://download.pytorch.org/whl/cpu
uv run jupyter lab
```

Блокнот лежит в `site/notebooks/extra/`, данные берёт из `site/data/`, кэш эмбеддингов пишет в `site/notebooks/extra/cache/` (папку можно удалить в любой момент — всё пересчитается). Первый полный запуск на ноутбуке — от нескольких десятков секунд до пары минут (плюс скачивание модели), повторный — несколько секунд.

In [1]:
# Импорты, версии и общие настройки
import hashlib
import json
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import torch
import transformers
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from transformers import AutoModel, AutoTokenizer

RANDOM_STATE = 42
DATA = Path('../../data')          # блокнот в site/notebooks/extra, данные в site/data
CACHE = Path('cache')              # кэш эмбеддингов рядом с блокнотом
MODEL_NAME = 'cointegrated/rubert-tiny2'
BATCH_SIZE = 32

torch.manual_seed(RANDOM_STATE)
print('Python', sys.version.split()[0])
for lib in (pd, np, sklearn, torch, transformers):
    print(f'{lib.__name__:<13}{lib.__version__}')

Python 3.12.14
pandas       3.0.6
numpy        2.5.3
sklearn      1.9.1
torch        2.14.0
transformers 5.17.0


## 1. Токенизатор: текст → кусочки → id

Токенизатор и модель всегда грузят **по одному имени**: номера кусочков у каждой модели свои.

In [2]:
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).eval()     # eval(): режим применения, dropout выключен

print('кусочков в словаре:', tok.vocab_size, '| предел длины:', tok.model_max_length)
for text in ['Курьер опоздал, кофе остыл.', 'Капучино и флэт уайт — отличные!', 'Бариста запомнил мой заказ']:
    print(tok.tokenize(text))

Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


кусочков в словаре: 83828 | предел длины: 2048
['Кур', '##ьер', 'опозда', '##л', ',', 'кофе', 'осты', '##л', '.']
['Кап', '##учи', '##но', 'и', 'фл', '##эт', 'у', '##айт', '[UNK]', 'отличные', '!']
['Бари', '##ста', 'запомнил', 'мой', 'заказ']


Что видно:

- частые слова — целиком, редкие — из кусочков; продолжение слова помечено `##`;
- регистр и знаки препинания токенизатор обрабатывает сам (посмотрите, остались ли заглавные буквы): отдельная очистка текста, как для TF-IDF, трансформеру не нужна, а иногда и вредна — знаки «!» и «?» тоже несут смысл.

Теперь пакет из текстов разной длины: дополнение `[PAD]` и маска.

In [3]:
texts = ['Очень вкусный раф!', 'Курьер опоздал на сорок минут, кофе остыл, крышка протекла.']
batch = tok(texts, padding=True, truncation=True, max_length=128, return_tensors='pt')
for i in range(len(texts)):
    print(tok.convert_ids_to_tokens(batch['input_ids'][i]))
print(batch['attention_mask'])

['[CLS]', 'Очень', 'вкусный', 'ра', '##ф', '!', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']
['[CLS]', 'Кур', '##ьер', 'опозда', '##л', 'на', 'сорок', 'минут', ',', 'кофе', 'осты', '##л', ',', 'крыш', '##ка', 'проте', '##кла', '.', '[SEP]']
tensor([[1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])


### Вывод по разделу 1

`padding=True` дополняет короткие тексты до длины самого длинного в пакете, `attention_mask` отмечает настоящие токены единицами. `[CLS]` и `[SEP]` токенизатор добавляет сам.

## 2. Векторы токенов и пулинг с маской

Модель возвращает вектор на каждую позицию, в том числе на заполнители. Вектор текста — среднее по настоящим токенам.

In [4]:
def mean_pool(hidden, mask):
    """Среднее векторов токенов только по позициям с mask == 1."""
    m = mask.unsqueeze(-1).to(hidden.dtype)              # тексты × токены × 1
    return (hidden * m).sum(dim=1) / m.sum(dim=1).clamp(min=1e-9)

with torch.no_grad():                                    # без градиентов: быстрее и меньше памяти
    hidden = model(**batch).last_hidden_state            # тексты × токены × 312
print('форма выхода модели:', tuple(hidden.shape))

pooled = mean_pool(hidden, batch['attention_mask'])
naive = hidden.mean(dim=1)                               # ошибка: среднее вместе с заполнителями
cos = torch.nn.functional.cosine_similarity(pooled[0], naive[0], dim=0).item()
print(f'короткий отзыв: косинус «с маской» и «в лоб» = {cos:.3f}')

форма выхода модели: (2, 19, 312)
короткий отзыв: косинус «с маской» и «в лоб» = 0.925


Проверка, ради которой маска и нужна: вектор отзыва, закодированного отдельно, должен совпасть с вектором того же отзыва в пакете с длинным соседом.

In [5]:
def embed_batch(texts):
    enc = tok(list(texts), padding=True, truncation=True, max_length=128, return_tensors='pt')
    with torch.no_grad():
        out = model(**enc).last_hidden_state
    e = mean_pool(out, enc['attention_mask'])
    e = torch.nn.functional.normalize(e, dim=1)          # длина 1: косинус = скалярное произведение
    return e.numpy()

alone = embed_batch([texts[0]])[0]
in_batch = embed_batch(texts)[0]
print('максимальная разница:', float(np.abs(alone - in_batch).max()))

максимальная разница: 5.960464477539063e-08


### Вывод по разделу 2

С маской вектор короткого отзыва не зависит от соседей по пакету (разница — на уровне погрешности вычислений, порядка 1e-6). Среднее «в лоб» даёт другой вектор, и тем сильнее, чем больше заполнителей.

## 3. Все отзывы — пакетами и с кэшем

Кодирование — самая медленная часть. Векторы сохраняем на диск и при следующем запуске читаем. Кэш должен «знать», для каких текстов и какой модели он посчитан: иначе после правки данных вы незаметно получите старые векторы. Поэтому имя файла содержит **отпечаток** (md5, глава 15) списка текстов, имени модели и параметров.

In [6]:
def fingerprint(texts, **params):
    h = hashlib.md5()
    h.update(json.dumps(params, sort_keys=True).encode('utf-8'))
    for t in texts:
        h.update(t.encode('utf-8'))
        h.update(b'\x00')                                 # разделитель: «аб»+«в» ≠ «а»+«бв»
    return h.hexdigest()[:12]


def embed_cached(texts, encode_fn, batch_size=BATCH_SIZE, **params):
    """Векторы для texts: из кэша, если он есть для ровно этих текстов и параметров, иначе — посчитать и сохранить."""
    texts = list(texts)
    CACHE.mkdir(exist_ok=True)
    path = CACHE / f'emb_{fingerprint(texts, batch_size=batch_size, **params)}.npy'
    if path.exists():
        E = np.load(path)
        if E.shape[0] == len(texts):
            print(f'из кэша: {path}')
            return E
    start = time.perf_counter()
    parts = [encode_fn(texts[i:i + batch_size]) for i in range(0, len(texts), batch_size)]
    E = np.vstack(parts).astype(np.float32)
    np.save(path, E)
    print(f'посчитано за {time.perf_counter() - start:.1f} с и сохранено: {path}')
    return E

In [7]:
reviews = pd.read_csv(DATA / 'reviews_text.csv')
y = (reviews['rating'] <= 3).astype(int)

E = embed_cached(reviews['text'], embed_batch, model=MODEL_NAME, max_length=128, pooling='mean')
print(E.shape, '| длины векторов:', np.linalg.norm(E, axis=1).round(3)[:5])

посчитано за 0.4 с и сохранено: cache/emb_5346636ab1a0.npy
(2400, 312) | длины векторов: [1. 1. 1. 1. 1.]


Запустите ячейку выше ещё раз: теперь векторы читаются из кэша за доли секунды. Если поменять хотя бы один отзыв, модель или `max_length`, отпечаток изменится и векторы пересчитаются.

### Вывод по разделу 3

2400 отзывов кодируются пакетами по 32; результат — матрица 2400 × 312 с векторами длины 1. Кэш с отпечатком данных и параметров экономит минуты на каждом перезапуске и не даёт перепутать версии.

## 4. Классификатор тональности на эмбеддингах

Матрица `E` — обычные числовые признаки. Сравним на одной и той же 5-кратной кросс-валидации логистическую регрессию на эмбеддингах и привычный TF-IDF.

In [8]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

f1_emb = cross_val_score(LogisticRegression(max_iter=2000), E, y, cv=cv, scoring='f1')
tfidf_model = make_pipeline(TfidfVectorizer(ngram_range=(1, 2)), LogisticRegression(max_iter=1000))
f1_tfidf = cross_val_score(tfidf_model, reviews['text'], y, cv=cv, scoring='f1')

print(f'эмбеддинги rubert-tiny2: F1 = {f1_emb.mean():.3f} ± {f1_emb.std():.3f}')
print(f'TF-IDF слов и биграмм:   F1 = {f1_tfidf.mean():.3f} ± {f1_tfidf.std():.3f}')

эмбеддинги rubert-tiny2: F1 = 0.930 ± 0.016
TF-IDF слов и биграмм:   F1 = 0.940 ± 0.015


### Вывод по разделу 4

✎ Запишите, что получилось у вас. Для сравнения, замер курса (torch 2.14, transformers 5.17): эмбеддинги 0.930 ± 0.016, TF-IDF 0.940 ± 0.015. На коротких шаблонных отзывах «Зерна» TF-IDF уже почти упирается в шум разметки, поэтому большого выигрыша от эмбеддингов ждать не стоит; на живых длинных текстах с синонимами и жаргоном разница бывает заметной. Сравнивайте по кросс-валидации на одних и тех же фолдах, а не «на глаз».

## 5. Поиск похожих отзывов по смыслу

Тот же поиск, что в разделе «Разберём вместе» урока, но на векторах трансформера. Запрос кодируется той же функцией; векторы уже нормированы, поэтому сходство — `E @ q`.

In [9]:
def search(query, k=5):
    q = embed_batch([query])[0]
    sim = E @ q
    top = np.argsort(-sim)[:k]
    return pd.DataFrame({'сходство': sim[top].round(3), 'отзыв': reviews.loc[top, 'text'].values})

for query in ['курьер опоздал', 'долго везли заказ', 'кофе остыл пока везли', 'дорого', 'не дорого']:
    print('запрос:', query)
    print(search(query).to_string(index=False), end='\n\n')

запрос: курьер опоздал
 сходство                                                                   отзыв
    0.803                                               Доставка опоздала на час.
    0.746                        Доставка опоздала на час, никто не поздоровался.
    0.743                              Доставка опоздала на час. Больше не приду.
    0.705                      Хожу сюда давно. Доставка опоздала на час. Обидно.
    0.700 Зашёл утром перед работой. Доставка опоздала на час, круассан чёрствый.

запрос: долго везли заказ
 сходство                                                                     отзыв
    0.731                                          Брали с собой. Перепутали заказ.
    0.716                                           Были сегодня. Перепутали заказ.
    0.716                                           Были сегодня. Перепутали заказ.
    0.709 Заказывала через приложение. Кофе привезли горячим, но сырник пересушены.
    0.709    Хожу сюда давно. Долго не

### Вывод по разделу 5

Замер курса (torch 2.14, transformers 5.19), сравните со своим:

- «курьер опоздал» — пять отзывов «Доставка опоздала на час», хотя слова «курьер» в них нет: здесь модель поймала смысл;
- «долго везли заказ» — промах: наверху отзывы с общими словами «заказ» и «привезли» («Перепутали заказ», «Кофе привезли горячим»), а первый отзыв про опоздавшую доставку — только на 74-м месте;
- «не дорого» — модель находит «Слишком дорого»: отрицание она почти не замечает.

✎ Запишите, что получилось у вас, и сравните с поиском на LSA из урока. Поиск по смыслу проверяют на десятке своих типичных запросов, прежде чем ему доверять.


## Попробуйте сами

1. Замените пулинг на вектор `[CLS]` (`out[:, 0]`) и сравните F1 из раздела 4. Не забудьте, что кэш различает параметры: передайте `pooling='cls'` в `embed_cached`.
2. Уберите маску из `mean_pool` (замените на `hidden.mean(dim=1)`) и повторите проверку из раздела 2 — насколько разошлись векторы?
3. Закодируйте `reviews_new.csv` (4800 отзывов) и найдите кластеры жалоб `KMeans` на векторах. Сколько времени занял первый запуск и сколько — повторный?